# Libraries

In [1]:
from typing import Optional
import torch
from torch import nn

# Ghost Batch Normalization

In [65]:
class GhostBatchNorm(nn.Module):
    """
    Ghost Batch Normalization (GBN) is a variant of batch normalization 
    that is used in the TabNet neural network architecture. 
    GBN aims to improve the performance of batch normalization 
    by addressing some of its limitations.

    Batch normalization is a technique commonly used in neural networks 
    to normalize the activations of each layer. 
    It helps in stabilizing the learning process and accelerating convergence. 
    However, traditional batch normalization calculates the statistics 
    (mean and variance) of the activations using only the examples in the current mini-batch. 
    This can lead to high variance in the estimated statistics, 
    especially when the mini-batch size is small.

    GBN addresses this limitation by introducing a "ghost" batch, 
    which is a larger virtual batch containing multiple mini-batches. 
    Instead of calculating the statistics based on a single mini-batch, 
    GBN computes them using both the current mini-batch and the ghost batch. 
    This way, the estimated statistics become more stable and reliable.

    The ghost batch is constructed by accumulating statistics 
    from multiple iterations during training. 
    It is typically much larger than a single mini-batch and contains 
    a representative sample of the entire training dataset. 
    By incorporating information from a larger batch, 
    GBN reduces the variance in the estimated statistics and improves their accuracy.
    """
    def __init__(self, 
                 features: int,
                 virtual_batch_size: Optional[int] = 5,
                 momentum: Optional[float] = 0.01
                 ):
        super().__init__()
        self.bn = nn.BatchNorm1d(features, momentum=momentum)
        self.features = features
        self.virtual_batch_size = virtual_batch_size
        
    def split_chunks(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        if self.virtual_batch_size > x.size(0):
            raise ValueError(
                f"Virtual batch size must be less than global batch size, "
                f"but 'virtual_batch_size' is {self.virtual_batch_size} "
                f"and global batch size is {x.size(0)}"
            )
        num_chunks = x.size(0) // self.virtual_batch_size
        return torch.chunk(x, num_chunks, 0)
    
    def batch_norm(self, chunks: tuple[torch.Tensor, ...]) -> list[torch.Tensor, ...]:
        return [
            self.bn(chunk) 
            for chunk in chunks
        ]
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        chunks = self.split_chunks(x)
        chunks = self.batch_norm(chunks)
        return torch.cat(chunks, dim=0)

# Transformers

In [64]:
"""
This module contains implementations of transformers for TabNet:
    - AttentiveTransformer
    - FeatTransformer
"""

__all__ = [
    "AttentiveTransformer",
    "FeatTransformer"
]

from typing import Literal, Optional
import torch
from torch import nn


class FeatTransformer(nn.Module):
    """
    Parameters
    ----------
    input_dim : int
        Input size
    output_dim : int
        Output_size
    shared_layers : torch.nn.ModuleList
        The shared block that should be common to every step
    n_glu_independent : int
        Number of independent GLU layers
    virtual_batch_size : int
        Batch size for Ghost Batch Normalization within GLU block(s)
    momentum : float
        Float value between 0 and 1 which will be used for momentum in batch norm
    """
    def __init__(self,
                 input_dim,
                 output_dim,
                 shared_layers,
                 n_glu_independent,
                 virtual_batch_size: Optional[int] = 128,
                 momentum: Optional[float] = 0.02,
                 ):
        super().__init__()


        params = {
            "n_glu": n_glu_independent,
            "virtual_batch_size": virtual_batch_size,
            "momentum": momentum,
        }

        if shared_layers is None:
            # no shared layers
            self.shared = torch.nn.Identity()
            is_first = True
        else:
            self.shared = GLU_Block(
                input_dim,
                output_dim,
                first=True,
                shared_layers=shared_layers,
                n_glu=len(shared_layers),
                virtual_batch_size=virtual_batch_size,
                momentum=momentum,
            )
            is_first = False

        if n_glu_independent == 0:
            # no independent layers
            self.specifics = torch.nn.Identity()
        else:
            spec_input_dim = input_dim if is_first else output_dim
            self.specifics = GLU_Block(
                spec_input_dim, output_dim, first=is_first, **params
            )

    def forward(self, x):
        x = self.shared(x)
        x = self.specifics(x)
        return x


class AttentiveTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        group_dim: int,
        group_matrix,
        virtual_batch_size: Optional[int] = 128,
        momentum: Optional[float] = 0.02,
        mask_type: Optional[Literal["sparsemax"]] = "sparsemax",
    ):
        """
        Initialize an attention transformer.

        Parameters
        ----------
        input_dim : int
            Input size
        group_dim : int
            Number of groups for features
        virtual_batch_size : int
            Batch size for Ghost Batch Normalization
        momentum : float
            Float value between 0 and 1 which will be used for momentum in batch norm
        mask_type : str
            Either "sparsemax" or "entmax" : this is the masking function to use
        """
        super(AttentiveTransformer, self).__init__()
        self.fc = Linear(input_dim, group_dim, bias=False)
        initialize_non_glu(self.fc, input_dim, group_dim)
        self.bn = GBN(
            group_dim, virtual_batch_size=virtual_batch_size, momentum=momentum
        )

        if mask_type == "sparsemax":
            # Sparsemax
            self.selector = sparsemax.Sparsemax(dim=-1)
        elif mask_type == "entmax":
            # Entmax
            self.selector = sparsemax.Entmax15(dim=-1)
        else:
            raise NotImplementedError(
                "Please choose either sparsemax" + "or entmax as masktype"
            )

    def forward(self, priors, processed_feat):
        x = self.fc(processed_feat)
        x = self.bn(x)
        x = torch.mul(x, priors)
        x = self.selector(x)
        return x